# 03 — Data Enrichment: Market Data, Technicals & News

This notebook enriches your portfolio holdings with:
- **Current & historical prices** (yfinance)
- **Technical indicators** — RSI, SMA 50/200, EMA 12/26, MACD, Bollinger Bands (pandas-ta)
- **Fundamentals** — P/E, market cap, dividend yield, 52-week range, sector (yfinance)
- **Recent news headlines** (yfinance)

The output is an enriched JSON file ready to feed into Claude for analysis in Phase 3.

**Prerequisites:**
- Run notebooks 01 and 02 first
- `portfolio_snapshot.json` must exist
- Run `uv sync` after pulling this update (new dependencies: yfinance, pandas-ta)

## Setup

In [1]:
import json
import os
import re
import warnings
from datetime import datetime, timedelta, timezone

import pandas as pd
import pandas_ta as ta
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Load portfolio snapshot from notebook 02
SNAPSHOT_FILE = os.path.join("..", "portfolio_snapshot.json")

if not os.path.exists(SNAPSHOT_FILE):
    raise FileNotFoundError(
        "portfolio_snapshot.json not found. Run notebook 02 first."
    )

with open(SNAPSHOT_FILE, "r") as f:
    portfolio = json.load(f)

holdings_df = pd.DataFrame(portfolio["holdings"])

def is_valid_ticker(ticker: str) -> bool:
    """Return True if ticker is fetchable by yfinance (stock, ETF, mutual fund, or option).
    Filters out CUSIPs (e.g. 92202V476), freeform strings (e.g. 'Pending activity'), etc.
    """
    if not ticker or " " in ticker:
        return False
    # Option format: -SYMBOL YYMMDD C/P STRIKE
    if ticker.startswith("-"):
        return bool(re.match(r"^-[A-Z]+\d{6}[CP]\d+", ticker))
    # Standard stock/ETF/mutual fund: 1-6 uppercase letters, optional dot + 1-2 letters
    return bool(re.match(r"^[A-Z]{1,6}(\.[A-Z]{1,2})?$", ticker))

def format_yf_option(ticker: str) -> str:
    """Convert Fidelity-style tickers like -JD270115C25 to yfinance-style JD270115C00025000."""
    match = re.match(r"-?([A-Z]+)(\d{6})([CP])(\d+(\.\d+)?)", ticker)
    if not match: return ticker
    symbol, date, cp, strike, _ = match.groups()
    strike_val = float(strike)
    strike_str = f"{int(strike_val * 1000):08d}"
    return f"{symbol}{date}{cp}{strike_str}"

# Get unique tickers (exclude cash, None, and unenrichable symbols)
all_raw = (
    holdings_df[holdings_df["type"] != "cash"]["ticker"]
    .dropna()
    .unique()
    .tolist()
)
skipped = [t for t in all_raw if not is_valid_ticker(t)]
# Deduplicate while preserving order (same ticker across multiple accounts)
seen: set = set()
tickers = [t for t in all_raw if is_valid_ticker(t) and not (t in seen or seen.add(t))]

if skipped:
    print(f"⚠️  Skipping {len(skipped)} unenrichable tickers: {skipped}")

# Add RSU tickers for price tracking even if not currently held
RSU_TICKERS = ["SNOW"]
for t in RSU_TICKERS:
    if t not in tickers:
        tickers.append(t)
        print(f"📌 Added RSU ticker {t} for price tracking")

print(f"✅ Loaded portfolio: {len(holdings_df)} positions")
print(f"📊 Unique tickers to enrich: {len(tickers)}")
print(f"   {tickers}")


⚠️  Skipping 2 unenrichable tickers: ['Pending activity', '92202V476']
📌 Added RSU ticker SNOW for price tracking
✅ Loaded portfolio: 44 positions
📊 Unique tickers to enrich: 36
   ['UPS', 'META', 'NVO', 'LULU', 'BABA', 'TSLA', 'ZETA', 'GRAB', 'O', 'AHRT', 'VICI', '-JD270115C25', '-JD270617C25', '-NVO270115C30', 'MRK', 'SNAP', 'UNH', 'PFE', '-SNAP280121C5', '-SNAP270115C4', 'MO', 'GIS', 'GLD', 'SCHD', 'SPMO', 'SLV', '-ADBE270115C280', '-PALL260618C135', '-CPER270115C33', '-ADBE270115C320', 'FXAIX', 'FPADX', 'VGSLX', 'FSPSX', 'FFTHX', 'SNOW']


## 1. Fetch Historical Prices (6 months daily)

In [2]:
LOOKBACK_DAYS = 200  # enough for SMA-200
start_date = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")

print(f"Downloading daily prices since {start_date}...\n")

price_data = {}  # ticker -> DataFrame
failed_tickers = []

def fetch_price_history(yf_symbol: str, is_option: bool) -> pd.DataFrame:
    """Fetch price history, with a shorter-period fallback for options."""
    df = yf.download(yf_symbol, start=start_date, progress=False, auto_adjust=True)
    if df.empty and is_option:
        # Some options (esp. ADR options like JD, NVO) have sparse history — try 6-month window
        df = yf.download(yf_symbol, period="6mo", progress=False, auto_adjust=True)
    if df.empty and is_option:
        # Last resort: just get recent data
        df = yf.download(yf_symbol, period="1mo", progress=False, auto_adjust=True)
    return df

for ticker in tickers:
    try:
        yf_symbol = format_yf_option(ticker)
        is_option = ticker.startswith("-")
        df = fetch_price_history(yf_symbol, is_option)
        if df.empty:
            print(f"  ⚠️  {ticker}: No data returned (may be delisted or invalid)")
            failed_tickers.append(ticker)
        else:
            # Flatten multi-level columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            # Drop rows with NaN close prices (partial/settling day data)
            df = df[df["Close"].notna()]
            if df.empty:
                print(f"  ⚠️  {ticker}: All close prices are NaN")
                failed_tickers.append(ticker)
                continue
            price_data[ticker] = df
            print(f"  ✅ {ticker}: {len(df)} days, latest close ${df['Close'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  ❌ {ticker}: {e}")
        failed_tickers.append(ticker)

print(f"\n📈 Price data fetched for {len(price_data)}/{len(tickers)} tickers")
if failed_tickers:
    print(f"⚠️  Failed/skipped: {failed_tickers}")

  ✅ UPS: 139 days, latest close $96.94


  ✅ META: 139 days, latest close $595.84


  ✅ NVO: 139 days, latest close $36.81


  ✅ LULU: 139 days, latest close $165.60


  ✅ BABA: 139 days, latest close $124.11


...

  ✅ -CPER270115C33: 91 days, latest close $4.50
  ✅ -ADBE270115C320: 97 days, latest close $16.75


  ✅ FXAIX: 138 days, latest close $230.06


  ✅ FPADX: 138 days, latest close $14.47
  ✅ VGSLX: 138 days, latest close $129.85


  ✅ FSPSX: 138 days, latest close $61.09


  ✅ FFTHX: 138 days, latest close $17.83
  ✅ SNOW: 139 days, latest close $171.01

📈 Price data fetched for 36/36 tickers


## 2. Calculate Technical Indicators

For each ticker, we compute:
- **SMA 50 / SMA 200** — trend (Golden Cross / Death Cross)
- **EMA 12 / EMA 26** — momentum
- **RSI (14)** — overbought (>70) / oversold (<30)
- **MACD** — trend direction and momentum
- **Bollinger Bands (20, 2)** — volatility and mean reversion

In [3]:
technicals = {}  # ticker -> dict of latest indicator values

for ticker, df in price_data.items():
    close = df["Close"]
    
    # Moving averages
    sma_50 = ta.sma(close, length=50)
    sma_200 = ta.sma(close, length=200)
    ema_12 = ta.ema(close, length=12)
    ema_26 = ta.ema(close, length=26)
    
    # RSI
    rsi = ta.rsi(close, length=14)
    
    # MACD
    macd_df = ta.macd(close, fast=12, slow=26, signal=9)
    
    # Bollinger Bands
    bbands = ta.bbands(close, length=20, std=2)
    
    latest_price = close.iloc[-1]
    
    tech = {
        "price": round(float(latest_price), 2),
        "sma_50": round(float(sma_50.iloc[-1]), 2) if sma_50 is not None and not sma_50.empty and pd.notna(sma_50.iloc[-1]) else None,
        "sma_200": round(float(sma_200.iloc[-1]), 2) if sma_200 is not None and not sma_200.empty and pd.notna(sma_200.iloc[-1]) else None,
        "ema_12": round(float(ema_12.iloc[-1]), 2) if ema_12 is not None and not ema_12.empty else None,
        "ema_26": round(float(ema_26.iloc[-1]), 2) if ema_26 is not None and not ema_26.empty else None,
        "rsi_14": round(float(rsi.iloc[-1]), 2) if rsi is not None and not rsi.empty and pd.notna(rsi.iloc[-1]) else None,
    }
    
    # MACD values
    if macd_df is not None and not macd_df.empty:
        cols = macd_df.columns
        tech["macd"] = round(float(macd_df[cols[0]].iloc[-1]), 4) if pd.notna(macd_df[cols[0]].iloc[-1]) else None
        tech["macd_signal"] = round(float(macd_df[cols[2]].iloc[-1]), 4) if pd.notna(macd_df[cols[2]].iloc[-1]) else None
        tech["macd_hist"] = round(float(macd_df[cols[1]].iloc[-1]), 4) if pd.notna(macd_df[cols[1]].iloc[-1]) else None
    
    # Bollinger Bands
    if bbands is not None and not bbands.empty:
        bb_cols = bbands.columns
        tech["bb_lower"] = round(float(bbands[bb_cols[0]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[0]].iloc[-1]) else None
        tech["bb_mid"] = round(float(bbands[bb_cols[1]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[1]].iloc[-1]) else None
        tech["bb_upper"] = round(float(bbands[bb_cols[2]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[2]].iloc[-1]) else None
    
    # Derived signals
    if tech["sma_50"] and tech["sma_200"]:
        tech["golden_cross"] = tech["sma_50"] > tech["sma_200"]
    if tech["rsi_14"]:
        tech["rsi_signal"] = (
            "overbought" if tech["rsi_14"] > 70
            else "oversold" if tech["rsi_14"] < 30
            else "neutral"
        )
    if tech.get("bb_lower") and tech.get("bb_upper"):
        tech["bb_position"] = (
            "below_lower" if latest_price < tech["bb_lower"]
            else "above_upper" if latest_price > tech["bb_upper"]
            else "within_bands"
        )
    
    # Price vs moving averages
    if tech["sma_50"]:
        tech["pct_above_sma50"] = round((latest_price / tech["sma_50"] - 1) * 100, 2)
    if tech["sma_200"]:
        tech["pct_above_sma200"] = round((latest_price / tech["sma_200"] - 1) * 100, 2)
    
    technicals[ticker] = tech

print(f"✅ Technicals computed for {len(technicals)} tickers")

✅ Technicals computed for 36 tickers


### Technicals Overview Table

In [4]:
tech_df = pd.DataFrame.from_dict(technicals, orient="index")
tech_df.index.name = "ticker"

# Show key columns
display_cols = ["price", "sma_50", "sma_200", "rsi_14", "rsi_signal", "macd_hist", "bb_position"]
available_cols = [c for c in display_cols if c in tech_df.columns]

tech_df[available_cols]

,price,sma_50,sma_200,rsi_14,rsi_signal,macd_hist,bb_position
ticker,,,,,,,
UPS,96.94,108.19,None,27.98,oversold,-1.0153,within_bands
META,595.84,649.55,None,33.20,neutral,-5.0276,below_lower
NVO,36.81,48.31,None,31.14,neutral,0.3903,within_bands
LULU,165.60,180.13,None,42.32,neutral,0.1148,within_bands
BABA,124.11,153.69,None,24.66,oversold,-0.4098,within_bands
TSLA,376.59,415.46,None,34.96,neutral,-1.2747,below_lower
ZETA,17.31,18.59,None,45.42,neutral,-0.0460,within_bands
GRAB,3.64,4.25,None,30.17,neutral,-0.0154,within_bands
O,62.64,62.94,None,39.00,neutral,-0.5243,below_lower


## 3. Fetch Fundamentals

In [5]:
fundamentals = {}  # ticker -> dict

for ticker in price_data.keys():
    try:
        info = yf.Ticker(format_yf_option(ticker)).info
        
        fundamentals[ticker] = {
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "market_cap": info.get("marketCap"),
            "pe_trailing": info.get("trailingPE"),
            "pe_forward": info.get("forwardPE"),
            "peg_ratio": info.get("pegRatio"),
            "price_to_book": info.get("priceToBook"),
            "dividend_yield": info.get("dividendYield"),
            "beta": info.get("beta"),
            "fifty_two_week_high": info.get("fiftyTwoWeekHigh"),
            "fifty_two_week_low": info.get("fiftyTwoWeekLow"),
            "avg_volume": info.get("averageVolume"),
            "revenue_growth": info.get("revenueGrowth"),
            "earnings_growth": info.get("earningsGrowth"),
            "profit_margin": info.get("profitMargins"),
            "return_on_equity": info.get("returnOnEquity"),
            "debt_to_equity": info.get("debtToEquity"),
            "free_cash_flow": info.get("freeCashflow"),
            "analyst_target": info.get("targetMeanPrice"),
            "analyst_rec": info.get("recommendationKey"),
            "short_description": info.get("longBusinessSummary", "")[:200],
        }
        print(f"  ✅ {ticker}: {fundamentals[ticker].get('sector', 'N/A')} | P/E {fundamentals[ticker].get('pe_trailing', 'N/A')}")
    except Exception as e:
        print(f"  ⚠️  {ticker}: {e}")

print(f"\n✅ Fundamentals fetched for {len(fundamentals)}/{len(price_data)} tickers")

  ✅ UPS: Industrials | P/E 14.7772875


  ✅ META: Communication Services | P/E 25.381817


  ✅ NVO: Healthcare | P/E 10.338484


  ✅ LULU: Consumer Cyclical | P/E 11.518415


  ✅ BABA: Consumer Cyclical | P/E 16.35573


...

  ✅ FPADX: None | P/E None


  ✅ VGSLX: None | P/E 30.790867


  ✅ FSPSX: None | P/E None


  ✅ FFTHX: None | P/E None


  ✅ SNOW: Technology | P/E None

✅ Fundamentals fetched for 36/36 tickers


In [6]:
fund_df = pd.DataFrame.from_dict(fundamentals, orient="index")
fund_df.index.name = "ticker"

display_cols = ["sector", "market_cap", "pe_trailing", "pe_forward", "dividend_yield", "beta", "analyst_rec", "analyst_target"]
available_cols = [c for c in display_cols if c in fund_df.columns]

fund_df[available_cols]

,sector,market_cap,pe_trailing,pe_forward,dividend_yield,beta,analyst_rec,analyst_target
ticker,,,,,,,,
UPS,Industrials,8.231107e+10,14.777287,12.177747,6.79,1.054,buy,113.07143
META,Communication Services,1.507527e+12,25.381817,16.609951,0.35,1.279,strong_buy,863.62933
NVO,Healthcare,1.640651e+11,10.338484,11.098162,5.01,0.265,buy,47.33628
LULU,Consumer Cyclical,1.965615e+10,11.518415,12.441283,NaN,1.013,hold,191.92458
BABA,Consumer Cyclical,2.963734e+11,16.355730,15.418193,0.84,NaN,strong_buy,198.97375
TSLA,Consumer Cyclical,1.413372e+12,355.334930,134.020420,NaN,1.926,buy,421.61365
ZETA,Technology,4.258042e+09,NaN,14.576021,NaN,1.280,buy,29.08333
GRAB,Technology,1.492575e+10,60.666668,24.863388,NaN,0.962,strong_buy,6.49538
O,Real Estate,5.752209e+10,52.538464,35.012474,5.16,0.774,hold,67.85000


## 4. Fetch Recent News

In [7]:
MAX_NEWS_PER_TICKER = 5

news_data = {}  # ticker -> list of headlines

for ticker in price_data.keys():
    try:
        match = re.match(r"^-?([A-Z]+)", ticker)
        underlying = match.group(1) if match else ticker
        yf_ticker = yf.Ticker(underlying)
        raw_news = yf_ticker.news or []
        
        headlines = []
        for item in raw_news[:MAX_NEWS_PER_TICKER]:
            headline = {
                "title": item.get("title", ""),
                "publisher": item.get("publisher", ""),
                "link": item.get("link", ""),
            }
            # Extract publish time if available
            pub_time = item.get("providerPublishTime")
            if pub_time:
                headline["published"] = datetime.fromtimestamp(pub_time).strftime("%Y-%m-%d %H:%M")
            headlines.append(headline)
        
        news_data[ticker] = headlines
        print(f"  ✅ {ticker}: {len(headlines)} articles")
    except Exception as e:
        print(f"  ⚠️  {ticker}: {e}")
        news_data[ticker] = []

print(f"\n📰 News fetched for {len(news_data)} tickers")


  ✅ UPS: 5 articles
  ✅ META: 5 articles


  ✅ NVO: 5 articles


  ✅ LULU: 5 articles
  ✅ BABA: 5 articles


  ✅ TSLA: 5 articles


  ✅ ZETA: 5 articles


...

  ✅ -CPER270115C33: 5 articles
  ✅ -ADBE270115C320: 5 articles


  ✅ FXAIX: 0 articles
  ✅ FPADX: 0 articles


  ✅ VGSLX: 5 articles
  ✅ FSPSX: 0 articles


  ✅ FFTHX: 0 articles


  ✅ SNOW: 5 articles

📰 News fetched for 36 tickers


In [8]:
# Preview news for first ticker
preview_ticker = list(news_data.keys())[0] if news_data else None

if preview_ticker and news_data[preview_ticker]:
    print(f"\n📰 Recent headlines for {preview_ticker}:")
    for article in news_data[preview_ticker]:
        date = article.get('published', 'N/A')
        print(f"  [{date}] {article['title']}")
        print(f"           — {article['publisher']}")
else:
    print("No news available to preview.")


📰 Recent headlines for UPS:
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 


## 5. Performance Metrics

In [9]:
performance = {}  # ticker -> return periods

for ticker, df in price_data.items():
    close = df["Close"]
    latest = close.iloc[-1]
    
    perf = {}
    
    # 1-day return
    if len(close) >= 2:
        perf["return_1d"] = round(float((latest / close.iloc[-2] - 1) * 100), 2)
    
    # 1-week return
    if len(close) >= 5:
        perf["return_1w"] = round(float((latest / close.iloc[-5] - 1) * 100), 2)
    
    # 1-month return
    if len(close) >= 21:
        perf["return_1m"] = round(float((latest / close.iloc[-21] - 1) * 100), 2)
    
    # 3-month return
    if len(close) >= 63:
        perf["return_3m"] = round(float((latest / close.iloc[-63] - 1) * 100), 2)
    
    # 6-month return
    if len(close) >= 126:
        perf["return_6m"] = round(float((latest / close.iloc[-126] - 1) * 100), 2)
    
    # 52-week high/low proximity
    high = fundamentals.get(ticker, {}).get("fifty_two_week_high")
    low = fundamentals.get(ticker, {}).get("fifty_two_week_low")
    if high:
        perf["pct_from_52w_high"] = round(float((latest / high - 1) * 100), 2)
    if low:
        perf["pct_from_52w_low"] = round(float((latest / low - 1) * 100), 2)
    
    # Analyst target upside/downside
    target = fundamentals.get(ticker, {}).get("analyst_target")
    if target:
        perf["analyst_upside_pct"] = round(float((target / latest - 1) * 100), 2)
    
    performance[ticker] = perf

perf_df = pd.DataFrame.from_dict(performance, orient="index")
perf_df.index.name = "ticker"

print("📊 Performance Summary:")
perf_df

📊 Performance Summary:


,return_1d,return_1w,return_1m,return_3m,return_6m,pct_from_52w_high,pct_from_52w_low,analyst_upside_pct
ticker,,,,,,,,
UPS,0.39,-0.47,-16.95,-3.71,18.96,-20.81,18.22,16.64
META,-1.79,-5.04,-9.05,-10.25,-23.27,-25.17,24.19,44.94
NVO,-0.73,-4.59,-22.37,-22.68,-40.05,-54.80,2.68,28.60
LULU,0.02,3.56,-11.59,-23.02,-2.37,-52.48,5.72,15.90
BABA,-0.63,-9.22,-19.64,-15.75,-23.77,-35.58,29.65,60.32
TSLA,-0.98,-4.80,-8.55,-22.09,-11.61,-24.51,75.77,11.96
ZETA,0.46,-3.35,6.13,-3.13,-19.45,-30.48,61.93,68.01
GRAB,-1.09,-2.93,-16.89,-26.02,-43.04,-45.02,8.33,78.44
O,-0.63,-2.79,-3.98,9.90,8.89,-7.80,23.53,8.32


## 6. Export Enriched Portfolio for Claude (Phase 3)

We combine everything into a single JSON file:
- Portfolio holdings (from Plaid)
- Technicals per ticker
- Fundamentals per ticker
- Performance metrics per ticker
- Recent news per ticker

In [10]:
enriched_portfolio = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "summary": portfolio["summary"],
    "holdings": portfolio["holdings"],
    "enrichment": {},
    "failed_tickers": failed_tickers,
}

# Combine all enrichment data per ticker
for ticker in price_data.keys():
    enriched_portfolio["enrichment"][ticker] = {
        "technicals": technicals.get(ticker, {}),
        "fundamentals": fundamentals.get(ticker, {}),
        "performance": performance.get(ticker, {}),
        "news": news_data.get(ticker, []),
    }

# Save
export_path = os.path.join("..", "enriched_portfolio.json")
with open(export_path, "w") as f:
    json.dump(enriched_portfolio, f, indent=2, default=str)

# File size
size_kb = os.path.getsize(export_path) / 1024

print(f"💾 Enriched portfolio saved to {export_path} ({size_kb:.1f} KB)")
print(f"")
print(f"   📋 Holdings:     {len(enriched_portfolio['holdings'])} positions")
print(f"   📈 Enriched:     {len(enriched_portfolio['enrichment'])} tickers")
print(f"   🔧 Technicals:   RSI, SMA 50/200, EMA, MACD, Bollinger Bands")
print(f"   📊 Fundamentals: P/E, market cap, beta, analyst targets, margins")
print(f"   📰 News:         {sum(len(v) for v in news_data.values())} total articles")
print(f"")
print(f"👉 This file will be fed to Claude for analysis in notebook 04 (Phase 3).")


💾 Enriched portfolio saved to ../enriched_portfolio.json (92.7 KB)

   📋 Holdings:     44 positions
   📈 Enriched:     36 tickers
   🔧 Technicals:   RSI, SMA 50/200, EMA, MACD, Bollinger Bands
   📊 Fundamentals: P/E, market cap, beta, analyst targets, margins
   📰 News:         155 total articles

👉 This file will be fed to Claude for analysis in notebook 04 (Phase 3).


## Quick Flags (Preview of What Claude Will Analyze)

A quick scan for positions that might warrant attention:

In [11]:
print("🚩 POSITIONS WORTH WATCHING:\n")

for ticker, tech in technicals.items():
    flags = []
    
    # RSI extremes
    if tech.get("rsi_signal") == "overbought":
        flags.append(f"⚠️  RSI {tech['rsi_14']} — overbought")
    elif tech.get("rsi_signal") == "oversold":
        flags.append(f"🟢 RSI {tech['rsi_14']} — oversold (potential buy)")
    
    # Death cross
    if tech.get("golden_cross") is False:
        flags.append("☠️  Death Cross (SMA 50 < SMA 200)")
    
    # Bollinger Band breakout
    if tech.get("bb_position") == "below_lower":
        flags.append("📉 Below lower Bollinger Band")
    elif tech.get("bb_position") == "above_upper":
        flags.append("📈 Above upper Bollinger Band")
    
    # Near 52-week low
    perf = performance.get(ticker, {})
    if perf.get("pct_from_52w_low") is not None and perf["pct_from_52w_low"] < 5:
        flags.append(f"🔻 Near 52-week low ({perf['pct_from_52w_low']:+.1f}%)")
    
    # Near 52-week high
    if perf.get("pct_from_52w_high") is not None and perf["pct_from_52w_high"] > -5:
        flags.append(f"🔺 Near 52-week high ({perf['pct_from_52w_high']:+.1f}%)")
    
    # Analyst upside/downside
    if perf.get("analyst_upside_pct") is not None:
        if perf["analyst_upside_pct"] > 20:
            flags.append(f"📊 Analyst target: {perf['analyst_upside_pct']:+.1f}% upside")
        elif perf["analyst_upside_pct"] < -10:
            flags.append(f"📊 Analyst target: {perf['analyst_upside_pct']:+.1f}% downside")
    
    if flags:
        print(f"  {ticker}:")
        for flag in flags:
            print(f"    {flag}")
        print()

print("\n💡 Claude will analyze these signals in context with fundamentals and news in Phase 3.")

🚩 POSITIONS WORTH WATCHING:

  UPS:
    🟢 RSI 27.98 — oversold (potential buy)

  META:
    📉 Below lower Bollinger Band
    📊 Analyst target: +44.9% upside

  NVO:
    🔻 Near 52-week low (+2.7%)
    📊 Analyst target: +28.6% upside

  BABA:
    🟢 RSI 24.66 — oversold (potential buy)
    📊 Analyst target: +60.3% upside

  TSLA:
    📉 Below lower Bollinger Band

  ZETA:
    📊 Analyst target: +68.0% upside

  GRAB:
    📊 Analyst target: +78.4% upside

  O:
    📉 Below lower Bollinger Band

  AHRT:
    📊 Analyst target: +25.1% upside

  VICI:
    🟢 RSI 26.9 — oversold (potential buy)
    📉 Below lower Bollinger Band
    🔻 Near 52-week low (+0.5%)
    📊 Analyst target: +27.8% upside

  -JD270115C25:
    🔻 Near 52-week low (+0.0%)
    🔺 Near 52-week high (+0.0%)

  -JD270617C25:
    🔻 Near 52-week low (+0.0%)
    🔺 Near 52-week high (+0.0%)

  -NVO270115C30:
    🔻 Near 52-week low (+3.7%)
    🔺 Near 52-week high (-0.0%)

  SNAP:
    🟢 RSI 28.33 — oversold (potential buy)
    🔻 Near 52-week l